In [2]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb

torch_nb.print_setup()

class ConvAddRelu(nn.Module):
    def __init__(self, cin=3, cout=8, k=3):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, k, padding=k // 2, bias=False)
        self.bias_vec = nn.Parameter(torch.zeros(cout))

    def forward(self, x):
        y = self.conv(x)
        y = y + self.bias_vec.view(1, -1, 1, 1)
        return F.relu(y)


Repository root -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon
torch-mlir-opt  -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/third-party/torch-mlir/build/bin/torch-mlir-opt
Artifacts dir   -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts


In [3]:
import torch
from torch_mlir import fx

conv_module = ConvAddRelu().eval()
example_input = torch.randn(1, 3, 16, 16, dtype=torch.float32)

torch_module = fx.export_and_import(conv_module, example_input, func_name="kernel")
torch_ir = torch_module.operation.get_asm()

torch_file = torch_nb.ARTIFACTS_DIR / "conv_add_relu_torch.mlir"
torch_file.write_text(torch_ir)

print(f"Wrote Torch dialect IR → {torch_file.resolve()}")
print(torch_ir)

Wrote Torch dialect IR → /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/conv_add_relu_torch.mlir
module {
  func.func @kernel(%arg0: !torch.vtensor<[1,3,16,16],f32>) -> !torch.vtensor<[1,8,16,16],f32> {
    %0 = torch.vtensor.literal(dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>) : !torch.vtensor<[8,3,3,3],f32>
    %none = torch.constant.none
    %int1 = torch.constant.int 1
    %int1_0 = torch.constant.int 1
    %1 = torch.prim.ListConstruct %int1, %int1_0 : (!torch.int, !torch.int) -> !torch.list<int>
    %int1_1 = torch.constant.int 1
    %int1_2 = torch.constant.int 1
    %2 = torch.prim.ListConstruct %int1_1, %int1_2 : (!torch.int, !torch.int) -> !torch.list<int>
    %int1_3 = torch.constant.int 1
    %int1_4 = torch.constant.int 1
    %3 = torch.prim.ListConstruct %int1_3, %int1_4 : (!torch.int, !torch.int) -> !torch.list<int>
    %false = torch.constant.bool false
    %int0 = torch.constant.int 0
    %int0_5 = torch.constant.int 0

In [4]:
from pathlib import Path

pipeline = (
    "builtin.module("
    "torch-function-to-torch-backend-pipeline,"
    "torch-backend-to-linalg-on-tensors-backend-pipeline,"
    "torch-verify-linalg-on-tensors-backend-contract"
    ")"
)

torch_ir = torch_nb.ARTIFACTS_DIR / "conv_add_relu_torch.mlir"
linalg_ir = torch_nb.ARTIFACTS_DIR / "conv_add_relu_linalg.mlir"

torch_nb.run(
    [
        torch_nb.torch_mlir_opt,
        torch_ir,
        f"-pass-pipeline={pipeline}",
        "-o",
        linalg_ir,
    ]
)

print(f"Wrote Linalg IR → {linalg_ir}")
linalg_txt = linalg_ir.read_text()
print(linalg_txt)

Wrote Linalg IR → /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/conv_add_relu_linalg.mlir
#map = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map1 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %1 = linalg.fill ins(%cst_0 : f32) outs(%0 : tensor<1x8x16x16xf32>) -> tensor<1x8x16x16xf32>
    %2 = linalg.conv_2d_nchw_fchw {dilations = dense<1> : vector<2xi64>, strides = dense<1>

In [5]:
from tutorial._infra import cinm_frontend as cinm_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-cleanup-linalg,im2col-to-matmul)"
    ")"
)

conv_add_relu_linalg_im2col_clean = torch_nb.ARTIFACTS_DIR / "conv_add_relu_linalg_im2col_clean.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        linalg_ir,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_linalg_im2col_clean,
    ]
)

conv_add_relu_linalg_im2col_clean_txt = conv_add_relu_linalg_im2col_clean.read_text()
print(conv_add_relu_linalg_im2col_clean_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %1

In [6]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-linalg-to-cinm)"
    ")"
)

conv_add_relu_cinm0 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm0.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_linalg_im2col_clean,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm0,
    ]
)

conv_add_relu_cinm0_txt = conv_add_relu_cinm0.read_text()
print(conv_add_relu_cinm0_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %1

In [7]:
cinm_pipeline = (
    "builtin.module("
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemm tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemv tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemm tile-sizes=32x32x32x32},"
    "cinm-annotate-tiles{ops=activate tile-sizes=32}"
    ")"
)

conv_add_relu_cinm1 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm1.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm0,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm1,
    ]
)

conv_add_relu_cinm1_txt = conv_add_relu_cinm1.read_text()
print(conv_add_relu_cinm1_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %1

In [8]:
cinm_pipeline = (
    "builtin.module("
    "cinm-gemm-to-gemv{split-dim=2}"
    ")"
)

conv_add_relu_cinm2 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm2.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm1,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm2,
    ]
)

conv_add_relu_cinm2_txt = conv_add_relu_cinm2.read_text()
print(conv_add_relu_cinm2_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %c

In [9]:
cinm_pipeline = (
    "builtin.module("
    "cinm-tiling"
    ")"
)

conv_add_relu_cinm3 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm3.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm2,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm3,
    ]
)

conv_add_relu_cinm3_txt = conv_add_relu_cinm3.read_text()
print(conv_add_relu_cinm3_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %c

In [10]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-decompose-accum)"
    ")"
)

conv_add_relu_cinm4 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm4.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm3,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm4,
    ]
)

conv_add_relu_cinm4_txt = conv_add_relu_cinm4.read_text()
print(conv_add_relu_cinm4_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %c

In [11]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-insert-quantization{ops=gemm,gemv qtype=i8 scale=0.03125 zp=0 rounding=nearest narrow-range=false})"
    ")"
)

conv_add_relu_cinm5 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm5.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm4,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm5,
    ]
)

conv_add_relu_cinm5_txt = conv_add_relu_cinm5.read_text()
print(conv_add_relu_cinm5_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<()[s0] -> (s0 floordiv 9)>
#map2 = affine_map<()[s0, s1] -> (s0 floordiv 16 + (s1 mod 9) floordiv 3)>
#map3 = affine_map<()[s0, s1] -> (s0 + s1 - (s0 floordiv 16) * 16 - (s1 floordiv 3) * 3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map5 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %c

In [12]:
cinm_pipeline = (
    "builtin.module("
    "lower-affine,"
    "func.func(cinm-relower)"
    ")"
)

conv_add_relu_cinm6 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm6.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm5,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm6,
    ]
)

conv_add_relu_cinm6_txt = conv_add_relu_cinm6.read_text()
print(conv_add_relu_cinm6_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map2 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  func.func @kernel(%arg0: tensor<1x3x16x16xf32>) -> tensor<1x8x16x16xf32> {
    %c0 = arith.constant 0 : index
    %cst = arith.constant dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>
    %cst_0 = arith.constant 0.000000e+00 : f32
    %cst_1 = arith.constant dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>
    %padded = tensor.pad %arg0 low[0, 0, 1, 1] high[0, 0, 1, 1] {
    ^bb0(%arg1: index, %arg2: index, %arg3: index, %arg4: index):
      tensor.yield %cst_0 : f32
    } : tensor<1x3x16x16xf32> to tensor<1x3x18x18xf32>
    %0 = tensor.empty() : tensor<1x8x16x16xf32>
    %collapsed = tensor.collapse_shape %cst [[0], [1, 2, 3]] : tensor<8x3x3x3xf32> into tensor<8x27xf32>
    %1 = tensor.empty() : tensor<1x27x256xf32>
    %2 = linalg.generic {indexing_maps = [#map], iterator_types =

In [13]:
cinm_pipeline = (
    "builtin.module("
    "func.func(linalg-generalize-named-ops,canonicalize,scf-for-loop-canonicalization),"
    "one-shot-bufferize{bufferize-function-boundaries}"
    ")"
)

conv_add_relu_cinm7 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm6,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm7,
    ]
)

conv_add_relu_cinm7_txt = conv_add_relu_cinm7.read_text()
print(conv_add_relu_cinm7_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> ()>
#map2 = affine_map<(d0) -> (d0)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  memref.global "private" constant @__constant_8xf32 : memref<8xf32> = dense_resource<torch_tensor_8_torch.float32> {alignment = 64 : i64}
  memref.global "private" constant @__constant_8x3x3x3xf32 : memref<8x3x3x3xf32> = dense_resource<torch_tensor_8_3_3_3_torch.float32> {alignment = 64 : i64}
  memref.global "private" constant @__constant_4xi64 : memref<4xi64> = dense<[1, 8, 16, 16]> {alignment = 64 : i64}
  memref.global "private" constant @__constant_1xi64 : memref<1xi64> = dense<2048> {alignment = 64 : i64}
  func.func @kernel(%arg0: memref<1x3x16x16xf32, strided<[?, ?, ?, ?], offset: ?>>) -> memref<1x8x16x16xf32> {
    %c1024 = arith.constant 1024 : index
    %c2048 = arith.constant 2048 : index
    %0 = memref.get_global @__constant_1xi64 : memref<1x

In [14]:
cinm_pipeline = (
    "builtin.module("
    "cinm-memory-cleanup,"
    "func.func(convert-cinm-to-cim,cim-mark-relower{ops=add,relu})"
    ")"
)

conv_add_relu_cinm8 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm8.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm7,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm8,
    ]
)

conv_add_relu_cinm8_txt = conv_add_relu_cinm8.read_text()
print(conv_add_relu_cinm8_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> ()>
#map2 = affine_map<(d0) -> (d0)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  memref.global "private" constant @__constant_8xf32 : memref<8xf32> = dense_resource<torch_tensor_8_torch.float32> {alignment = 64 : i64}
  memref.global "private" constant @__constant_8x3x3x3xf32 : memref<8x3x3x3xf32> = dense_resource<torch_tensor_8_3_3_3_torch.float32> {alignment = 64 : i64}
  memref.global "private" constant @__constant_4xi64 : memref<4xi64> = dense<[1, 8, 16, 16]> {alignment = 64 : i64}
  memref.global "private" constant @__constant_1xi64 : memref<1xi64> = dense<2048> {alignment = 64 : i64}
  func.func @kernel(%arg0: memref<1x3x16x16xf32, strided<[?, ?, ?, ?], offset: ?>>) -> memref<1x8x16x16xf32> {
    %c27 = arith.constant 27 : index
    %c8 = arith.constant 8 : index
    %c0_i8 = arith.constant 0 : i8
    %cst = arith.constant 0.000

In [16]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-cim-to-alpine,cim-cleanup-unsupported)"
    ")"
)

conv_add_relu_cinm9 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm9.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm8,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm9,
    ]
)

conv_add_relu_cinm9_txt = conv_add_relu_cinm9.read_text()
print(conv_add_relu_cinm9_txt)

#map = affine_map<(d0, d1, d2) -> (d0, d1, d2)>
#map1 = affine_map<(d0) -> ()>
#map2 = affine_map<(d0) -> (d0)>
#map3 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
#map4 = affine_map<(d0, d1, d2, d3) -> (d0, d1, 0, 0)>
module {
  memref.global "private" constant @__constant_8xf32 : memref<8xf32> = dense_resource<torch_tensor_8_torch.float32> {alignment = 64 : i64}
  memref.global "private" constant @__constant_8x3x3x3xf32 : memref<8x3x3x3xf32> = dense_resource<torch_tensor_8_3_3_3_torch.float32> {alignment = 64 : i64}
  memref.global "private" constant @__constant_4xi64 : memref<4xi64> = dense<[1, 8, 16, 16]> {alignment = 64 : i64}
  memref.global "private" constant @__constant_1xi64 : memref<1xi64> = dense<2048> {alignment = 64 : i64}
  func.func @kernel(%arg0: memref<1x3x16x16xf32, strided<[?, ?, ?, ?], offset: ?>>) -> memref<1x8x16x16xf32> {
    %c27 = arith.constant 27 : index
    %c8 = arith.constant 8 : index
    %c0_i8 = arith.constant 0 : i8
    %cst = arith.constant 0.000

In [27]:
cinm_pipeline = (
    "builtin.module("
    "convert-alpine-to-func,"
    "func.func(convert-linalg-to-loops),"
    "lower-affine,"
    "convert-scf-to-cf,"
    "expand-strided-metadata,"
    "convert-vector-to-llvm,"
    "convert-math-to-llvm,"
    "convert-arith-to-llvm,"
    "convert-index-to-llvm,"
    "convert-to-llvm,"
    "func.func(llvm-request-c-wrappers),"
    "reconcile-unrealized-casts,"
    "canonicalize"
    ")"
)

conv_add_relu_cinm10 = torch_nb.ARTIFACTS_DIR / "conv_add_relu_cinm10.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        conv_add_relu_cinm9,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        conv_add_relu_cinm10,
    ]
)

conv_add_relu_cinm10_txt = conv_add_relu_cinm10.read_text()
print(conv_add_relu_cinm10_txt)

module {
  llvm.func @memrefCopy(i64, !llvm.ptr, !llvm.ptr)
  llvm.func @malloc(i64) -> !llvm.ptr
  llvm.mlir.global private constant @__constant_8xf32(dense_resource<torch_tensor_8_torch.float32> : tensor<8xf32>) {addr_space = 0 : i32, alignment = 64 : i64} : !llvm.array<8 x f32>
  llvm.mlir.global private constant @__constant_8x3x3x3xf32(dense_resource<torch_tensor_8_3_3_3_torch.float32> : tensor<8x3x3x3xf32>) {addr_space = 0 : i32, alignment = 64 : i64} : !llvm.array<8 x array<3 x array<3 x array<3 x f32>>>>
  llvm.mlir.global private constant @__constant_4xi64(dense<[1, 8, 16, 16]> : tensor<4xi64>) {addr_space = 0 : i32, alignment = 64 : i64} : !llvm.array<4 x i64>
  llvm.mlir.global private constant @__constant_1xi64(dense<2048> : tensor<1xi64>) {addr_space = 0 : i32, alignment = 64 : i64} : !llvm.array<1 x i64>
  llvm.func @alpine_alloc_tile(i64, i64) -> i32 attributes {sym_visibility = "nested"}
  llvm.func @alpine_quantize_r2(!llvm.ptr, !llvm.ptr, i64, i64, i64, i64, i64, !llvm